#Querying Data Files

### Self-describing formats

JSON

In [0]:
%sql
SELECT * 
FROM json.`/Volumes/workspace/default/volume/students.json`; -- Einzeldatei

email,gpa,profile,student_id,updated
anna@example.com,3.9,full-time,1,2025-08-01
ben@example.com,3.2,part-time,2,2025-08-03
clara@example.com,3.7,exchange,3,2025-08-06


In [0]:
%sql
SELECT * 
FROM json.`/Volumes/workspace/default/volume/*.json`; -- wildcard -->  liest alle JSON-Dateien in diesem Ordner

_corrupt_record,category,course_id,courses,email,enroll_id,gpa,instructor,price,profile,quantity,student_id,timestamp,title,total,updated
null,null,null,null,e@aol.com,null,1.95,null,null,"{""first_name"":""Susana"",""last_name"":""Gonnely"",""gender"":""Female"",""address"":{""street"":""760 Express Court"",""city"":""Obrenovac"",""country"":""Serbia""}}",null,null,null,null,null,null
null,null,null,null,r@nb.com,null,3.33,null,null,"{""first_name"":""Ronna"",""last_name"":""Gonning"",""gender"":""Non-binary"",""address"":{""street"":""48 Grim Way"",""city"":""Metsomtholaba"",""country"":""Botswana""}}",null,null,null,null,null,null
null,null,null,null,r@gov.com,null,1.08,null,null,"{""first_name"":""Reade"",""last_name"":""Goode"",""gender"":""Male"",""address"":{""street"":""975 Mendota Center"",""city"":""Seabra"",""country"":""Brazil""}}",null,null,null,null,null,null
null,null,null,null,r@sk.com,null,1.97,null,null,"{""first_name"":""Row"",""last_name"":""Goodier"",""gender"":""Female"",""address"":{""street"":""..."",""city"":""..."",""country"":""...""}}",null,null,null,null,null,null
null,null,null,List(101),null,e001,null,null,null,null,1,1,2025-08-01T09:00:00,null,850,null
null,null,null,"List(102, 103)",null,e002,null,null,null,null,2,2,2025-08-02T10:00:00,null,1800,null
null,null,null,List(103),null,e003,null,null,null,null,1,3,2025-08-03T11:30:00,null,600,null
null,Technology,101,null,null,null,null,Dr. Smith,850,null,null,null,null,Data Engineering,null,null
null,AI,102,null,null,null,null,Dr. Lee,1200,null,null,null,null,Machine Learning,null,null
null,Philosophy,103,null,null,null,null,Dr. Kim,600,null,null,null,null,Ethics in AI,null,null


Parquet

In [0]:
%sql
SELECT * 
FROM parquet.`/Volumes/workspace/default/volume/*.parquet`;

id,name,price
1,Tesla,50000
2,VW,31000
3,Audi,76000


Delta

In [0]:
# Zielpfad im Volume (Delta schreibt in Ordner)
target_dir = "/Volumes/workspace/default/volume/sample_delta"

# 1) Beispiel-DataFrame
data = [
    (1, "Tesla", 50000),
    (2, "VW",  31000),
    (3, "Audi",  76000),
]
df = spark.createDataFrame(data, ["id", "name", "price"])

# 2) Als Delta ins Volume schreiben (Overwrite, falls vorhanden)
(df.write
   .format("delta")
   .mode("overwrite")
   .save(target_dir))

# Delta schreibt im Zielpfad eine "part-0000-.....parquet" Datei
# Beispiel:
#/Volumes/workspace/default/volume/part-00000-d92b98b7-b6ff-4e05-96a2-fc209758bbe5.c000.snappy.parquet

In [0]:
%sql
SELECT * 
FROM delta.`/Volumes/workspace/default/volume/sample_delta`;

id,name,price
1,Tesla,50000
2,VW,31000
3,Audi,76000


## CSV (Non-self-describing formats)

In [0]:
%sql
SELECT * 
FROM csv.`/Volumes/workspace/default/volume/sample_data.csv`;

_c0
id;name;age;city
1;Alice;25;Zurich
2;Bob;30;Geneva
3;Charlie;35;Bern


Lesen über Temporäre Views

In [0]:
%sql
-- State of the Art: Daten direkt in den Unity Catalog laden
CREATE OR REPLACE TEMPORARY VIEW sample_data
AS SELECT * FROM read_files(
  '/Volumes/workspace/default/volume/sample_data.csv',
  format => 'csv',
  header => true,
  sep => ';'
);

In [0]:
%sql
--Alternative Vairante (veraltet)
/*
CREATE OR REPLACE TEMPORARY VIEW sample_data
USING CSV
OPTIONS (
  'path'='/Volumes/workspace/default/volume/sample_data.csv',
  'delimiter'=';',
  'header'='true'
);
*/

In [0]:
%sql
SELECT * FROM sample_data;

id,name,age,city,_rescued_data
1,Alice,25,Zurich,null
2,Bob,30,Geneva,null
3,Charlie,35,Bern,null


### Ganze Verzeichnisse lesen

In [0]:
%sql
SELECT * 
FROM json.`/Volumes/workspace/default/volume/`;

_corrupt_record,category,course_id,courses,email,enroll_id,gpa,instructor,price,profile,quantity,student_id,timestamp,title,total,updated
"Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,LotConfig,LandSlope,Neighborhood,Condition1,Condition2,BldgType,HouseStyle,OverallQual,OverallCond,YearBuilt,YearRemodAdd,RoofStyle,RoofMatl,Exterior1st,Exterior2nd,MasVnrType,MasVnrArea,ExterQual,ExterCond,Foundation,BsmtQual,BsmtCond,BsmtExposure,BsmtFinType1,BsmtFinSF1,BsmtFinType2,BsmtFinSF2,BsmtUnfSF,TotalBsmtSF,Heating,HeatingQC,CentralAir,Electrical,1stFlrSF,2ndFlrSF,LowQualFinSF,GrLivArea,BsmtFullBath,BsmtHalfBath,FullBath,HalfBath,BedroomAbvGr,KitchenAbvGr,KitchenQual,TotRmsAbvGrd,Functional,Fireplaces,FireplaceQu,GarageType,GarageYrBlt,GarageFinish,GarageCars,GarageArea,GarageQual,GarageCond,PavedDrive,WoodDeckSF,OpenPorchSF,EnclosedPorch,3SsnPorch,ScreenPorch,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"1,60,RL,65,8450,Pave,NA,Reg,Lvl,AllPub,Inside,Gtl,CollgCr,Norm,Norm,1Fam,2Story,7,5,2003,2003,Gable,CompShg,VinylSd,VinylSd,BrkFace,196,Gd,TA,PConc,Gd,TA,No,GLQ,706,Unf,0,150,856,GasA,Ex,Y,SBrkr,856,854,0,1710,1,0,2,1,3,1,Gd,8,Typ,0,NA,Attchd,2003,RFn,2,548,TA,TA,Y,0,61,0,0,0,0,NA,NA,NA,0,2,2008,WD,Normal,208500",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"2,20,RL,80,9600,Pave,NA,Reg,Lvl,AllPub,FR2,Gtl,Veenker,Feedr,Norm,1Fam,1Story,6,8,1976,1976,Gable,CompShg,MetalSd,MetalSd,None,0,TA,TA,CBlock,Gd,TA,Gd,ALQ,978,Unf,0,284,1262,GasA,Ex,Y,SBrkr,1262,0,0,1262,0,1,2,0,3,1,TA,6,Typ,1,TA,Attchd,1976,RFn,2,460,TA,TA,Y,298,0,0,0,0,0,NA,NA,NA,0,5,2007,WD,Normal,181500",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"3,60,RL,68,11250,Pave,NA,IR1,Lvl,AllPub,Inside,Gtl,CollgCr,Norm,Norm,1Fam,2Story,7,5,2001,2002,Gable,CompShg,VinylSd,VinylSd,BrkFace,162,Gd,TA,PConc,Gd,TA,Mn,GLQ,486,Unf,0,434,920,GasA,Ex,Y,SBrkr,920,866,0,1786,1,0,2,1,3,1,Gd,6,Typ,1,TA,Attchd,2001,RFn,2,608,TA,TA,Y,0,42,0,0,0,0,NA,NA,NA,0,9,2008,WD,Normal,223500",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"4,70,RL,60,9550,Pave,NA,IR1,Lvl,AllPub,Corner,Gtl,Crawfor,Norm,Norm,1Fam,2Story,7,5,1915,1970,Gable,CompShg,Wd Sdng,Wd Shng,None,0,TA,TA,BrkTil,TA,Gd,No,ALQ,216,Unf,0,540,756,GasA,Gd,Y,SBrkr,961,756,0,1717,1,0,1,0,3,1,Gd,7,Typ,1,Gd,Detchd,1998,Unf,3,642,TA,TA,Y,0,35,272,0,0,0,NA,NA,NA,0,2,2006,WD,Abnorml,140000",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"5,60,RL,84,14260,Pave,NA,IR1,Lvl,AllPub,FR2,Gtl,NoRidge,Norm,Norm,1Fam,2Story,8,5,2000,2000,Gable,CompShg,VinylSd,VinylSd,BrkFace,350,Gd,TA,PConc,Gd,TA,Av,GLQ,655,Unf,0,490,1145,GasA,Ex,Y,SBrkr,1145,1053,0,2198,1,0,2,1,4,1,Gd,9,Typ,1,TA,Attchd,2000,RFn,3,836,TA,TA,Y,192,84,0,0,0,0,NA,NA,NA,0,12,2008,WD,Normal,250000",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"6,50,RL,85,14115,Pave,NA,IR1,Lvl,AllPub,Inside,Gtl,Mitchel,Norm,Norm,1Fam,1.5Fin,5,5,1993,1995,Gable,CompShg,VinylSd,VinylSd,None,0,TA,TA,Wood,Gd,TA,No,GLQ,732,Unf,0,64,796,GasA,Ex,Y,SBrkr,796,566,0,1362,1,0,1,1,1,1,TA,5,Typ,0,NA,Attchd,1993,Unf,2,480,TA,TA,Y,40,30,0,320,0,0,NA,MnPrv,Shed,700,10,2009,WD,Normal,143000",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"7,20,RL,75,10084,Pave,NA,Reg,Lvl,AllPub,Inside,Gtl,Somerst,Norm,Norm,1Fam,1Story,8,5,2004,2005,Gable,CompShg,VinylSd,VinylSd,Stone,186,Gd,TA,PConc,Ex,TA,Av,GLQ,1369,Unf,0,317,1686,GasA,Ex,Y,SBrkr,1694,0,0,1694,1,0,2,0,3,1,Gd,7,Typ,1,Gd,Attchd,2004,RFn,2,636,TA,TA,Y,255,57,0,0,0,0,NA,NA,NA,0,8,2007,WD,Normal,307000",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"8,60,RL,NA,10382,Pave,NA,IR1,Lvl,AllPub,Corner,Gtl,NWAmes,PosN,Norm,1Fam,2Story,7,6,1973,1973,Gable,CompShg,HdBoard,HdBoard,Stone,240,TA,TA,CBlock,Gd,TA,Mn,ALQ,859,BLQ,32,216,1107,GasA,E